# Design Summary
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

The final rollup of the design pipeline. Everything printed here is
**read from the `out/` handoffs the preceding fourteen notebooks wrote**
(plus one sizing-loop re-run to echo the closure, same pattern as every
notebook since NB2) — this notebook adds no new physics and writes no
handoff. It exists so a reader (or a release snapshot) has one place with:

1. the converged design point and the as-selected reality next to it,
2. the control / vibration / thermal / drag margins in one table,
3. the frozen COTS hardware list, and
4. **every standing finding** the pipeline carries, collected
   programmatically from the handoffs — nothing here is hand-maintained.

## Inputs

Everything in `out/` (NB1–NB14). No outputs.

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point
from conceptual_design import design_summary as ds
from conceptual_design.plots import plot_budget_margins

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

# Section 1 — Design Point at a Glance

Re-run the sizing loop from `config/` and cross-check it against the
frozen-hardware echo (`out/components.yaml`) — a mismatch means stale
handoffs, not new physics.

---

In [ ]:
# -- Re-run the design point; collect every handoff (this NB writes nothing) ---
inp, result = solve_design_point(CONFIG_PATH)

af       = load_handoff(OUT_PATH, "airfoil")
vanes    = load_handoff(OUT_PATH, "control_vanes")
ail      = load_handoff(OUT_PATH, "aileron")
ail_cots = load_handoff(OUT_PATH, "aileron_cots")
vib      = load_handoff(OUT_PATH, "vibration")
vib_cots = load_handoff(OUT_PATH, "vibration_cots")
fus      = load_handoff(OUT_PATH, "fuselage")
fus_cots = load_handoff(OUT_PATH, "fuselage_cots")
thermal  = load_handoff(OUT_PATH, "thermal")
elec     = load_handoff(OUT_PATH, "electrical")
massprop = load_handoff(OUT_PATH, "mass_properties")
comp     = load_handoff(OUT_PATH, "components")

# handoff freshness: the COTS freeze must echo this run's closure
assert abs(comp["design_point"]["MTOW_kg"] - result.m_total_kg) < 5e-3, (
    "out/components.yaml design-point echo != sizing re-run -- stale handoffs")

ds.print_glance(result, inp.mission, inp.rotor, inp.wf, af, elec, fus,
                fus_cots, massprop, comp)

# Section 2 — Mass: Closure Allocations vs. As-Selected Hardware

Same idiom as the NB11 budget chart (bar = actual hardware, tick =
allocation, red = over), for the five COTS groups of the freeze. The +/-
labels carry the numbers so the verdict never rides on color alone; the
all-up rollup (which would dwarf the group bars) is reported as text.

---

In [ ]:
labels = {
    "avionics_bay": "avionics bay",
    "esc":          "ESC",
    "motor_fan":    "EDF motor + prop",
    "servo_each":   "servo (each)",
    "battery":      "battery pack",
}
plot_budget_margins(
    comp["budgets"], ds.BUDGET_GROUPS, labels,
    "As-selected hardware vs mass-closure allocations\n"
    "(bar = actual, tick = allocation; red = over)",
    FIG_DIR / "design_summary_mass.png")

ds.print_budget_lines(comp["budgets"], fus_cots)

# Section 3 — Margins in One Table

Control authority (hover and cruise), servo torque, vibration isolation,
thermal paths, and the cruise drag budget — each against the requirement
its own notebook checked, with the notebook that owns it.

---

In [ ]:
rows = ds.margins_table(vanes, ail, ail_cots, vib_cots, thermal, fus_cots,
                        af, inp.aero)
ds.print_margins_table(rows)

# Section 4 — Frozen COTS Hardware

The procurement list as frozen by NB11 (`out/components.yaml`); `frozen`
means pinned via `selection.frozen` rather than auto-selected. After
procurement: weigh the parts, fix the `EST` entries, pin the remaining
ids, update the regression pins — the diff is the procurement record.

---

In [ ]:
display(ds.selected_hardware_table(comp))

# Section 5 — Standing Findings

Collected **programmatically** from the handoffs — this list cannot go
stale. Each finding indicts an *assumption* (a weight fraction, a
packaging density, an allocation), not a part: the selections were
already the lightest physics allows.

---

In [ ]:
findings = ds.collect_findings(comp, thermal, fus, fus_cots)
ds.print_findings(findings)

# Section 6 — Closing Summary

---

In [ ]:
ds.print_final_card(result, inp.rotor, elec, fus, fus_cots, comp, findings)